<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_Challenge_LangChain_OpenSource_StudentW8D1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [ ]:
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
except:
    print('CPU runtime detected.')

In [ ]:
!pip install -q langchain langchain-community transformers sentencepiece accelerate

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [ ]:

# TODO: import libs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain


In [ ]:

# TODO: choose a small model
model_name = "google/flan-t5-small"  # keep small for CPU


In [ ]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:

# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [ ]:
template = 'Rewrite this text to be simpler for beginners: {text}'
prompt = PromptTemplate(template=template, input_variables=['text'])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = 'LangChain helps you build LLM apps by composing prompts, models, and tools.'
rewritten = chain.run(text=sample_text)
print(rewritten)

## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [ ]:
from langchain.schema.runnable import RunnableLambda
from langchain import PromptTemplate

# Define prompt templates
summary_prompt = PromptTemplate(
    template="Summarize the following text concisely:\n\n{paragraph}",
    input_variables=["paragraph"],
)

bullets_prompt = PromptTemplate(
    template="Turn the following summary into exactly 3 short bullet points:\n\n{summary}",
    input_variables=["summary"],
)

In [ ]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

In [ ]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [ ]:

# TODO: build a simple conversation chain
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")
print("Turn 1:", reply1)
print("Turn 2:", reply2)


## Your observations
- **Latency**: Using `flan-t5-small` on CPU provides very low latency, typically sub-second responses.
- **Quality**: While efficient, the small model may produce very brief summaries or struggle with complex formatting instructions compared to larger models.
- **Quirks**: Small models like this are prone to repeating text if the prompt isn't specific, but they are excellent for basic text transformation tasks without needing a GPU.